### Updates the metadata by selecting the latest file on the metadata volume directory and after that applies MERGE INTO between the delta table "metadata" and the temp view of the latest file.

In [0]:
USE projects

In [0]:
CREATE TABLE IF NOT EXISTS workspace.projects.dim_sensors (
  id_sensor STRING NOT NULL,
  modelo STRING,
  ubicacion STRING,
  rango_max DOUBLE,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)

#### We ensure that we only take the latest file to create the temp view, just in case something ocurred at the source and we get more than 1 extra files when running this job

In [0]:
%python
try:
    # Find all the files in the directory
    files = dbutils.fs.ls("/Volumes/workspace/projects/iot_devices/iot_metadata/")

    # Sort by date
    latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

    df_latest = spark.read.parquet(latest_file.path)
    df_latest.createOrReplaceTempView("v_latest_metadata")
except IndexError:
    print("No new metadata file found")
except Exception as e:
    print(f"Error: {str(e)}")

dbutils.jobs.taskValues.set(key="latest_file_name", value=latest_file.name)

In [0]:
MERGE INTO workspace.projects.dim_sensors AS target
USING v_latest_metadata AS source
ON target.id_sensor = source.id_sensor
WHEN MATCHED THEN
  UPDATE SET 
    target.modelo = source.modelo,
    target.ubicacion = source.ubicacion,
    target.rango_max = source.rango_max,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (id_sensor, modelo, ubicacion, rango_max, created_at, updated_at)
  VALUES (
    source.id_sensor, 
    source.modelo, 
    source.ubicacion, 
    source.rango_max, 
    current_timestamp(),
    current_timestamp()
  );

In [0]:
SELECT * FROM workspace.projects.dim_sensors